##Multi-query Retrieval

---
Multi-Query Retrieval improves information retrieval by generating multiple rephrased queries and combining their results to increase coverage and accuracy.


###Step 1: Install Dependencies
These libraries help us interact with LLMs, generate embeddings, and store vectors.

In [ ]:
!pip install --quiet langchain-openai langchain-chroma pydantic


###Step 2: Imports & API Config
Allows our code to communicate with the OpenAI-compatible backend
* ChatOpenAI → to understand user query

* OpenAIEmbeddings → to convert text to vectors

* Chroma → vector database

* Pydantic → enforce structured output

In [ ]:
import os
from typing import List
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document



In [ ]:
os.environ["OPENAI_API_KEY"]="YOUR_API_KEY"
os.environ["OPENAI_API_BASE"]="https://apidev.navigatelabsai.com"

###Step 3: Documents
Content is used for semantic search, metadata can be used later for filtering

In [ ]:
docs=[
    Document(
        page_content="Scientists clone dinosaurs, chaos follows.",
        metadata={"year": 1993, "rating": 7.7, "genre": "sci-fi"}
    ),
    Document(
        page_content="A dream within a dream heist.",
        metadata={"year": 2010, "rating": 8.2, "genre": "sci-fi"}
    ),
    Document(
        page_content="Toys come alive when humans are away.",
        metadata={"year": 1995, "rating": 8.3, "genre": "animated"}
    ),
    Document(
        page_content="Detectives hunt a serial killer.",
        metadata={"year": 1995, "rating": 8.6, "genre": "crime"}
    ),
]


###Step 4: Embeddings + Vector Store
* Convert text into vectors
* Vector representations allow semantic similarity search.

In [ ]:
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://apidev.navigatelabsai.com"
)

vectorstore=Chroma.from_documents(docs, embeddings)

###Step 5: Multi-Query Generator
Ensures the LLM returns structured, predictable results.

In [ ]:
class MultiQueryOutput(BaseModel):
    queries: List[str]=Field(
        description="Different rephrased versions of the original query"
    )
llm=ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0,
    base_url="https://apidev.navigatelabsai.com"
)

query_generator=llm.with_structured_output(MultiQueryOutput)


###Step 6: Multi-Query Search Function
Converts one user query into multiple semantically similar queries.

In [ ]:
def multi_query_search(user_prompt, k=2):
    print("User Query:", user_prompt)

    # Step A: Generate multiple query variations
    mq_prompt=f"""
    Generate 3 different rephrased search queries
    that capture the meaning of:
    "{user_prompt}"
    """

    parsed=query_generator.invoke(mq_prompt)
    queries=parsed.queries

    print("\nGenerated Queries:")
    for q in queries:
        print("-", q)

    all_results=[] # Step B: Run vector search for each query

    for q in queries:
        results=vectorstore.similarity_search(q, k=k)
        all_results.extend(results)

    unique_docs={doc.page_content: doc for doc in all_results} # Step C: Deduplicate results

    return list(unique_docs.values())


###Step 7: Run It
Demonstrates how multiple query reformulations improve retrieval

In [ ]:
results=multi_query_search(
    "I want a sci-fi movie about dinosaurs",
    k=2
)

print("\n--- FINAL RESULTS ---")
for r in results:
    print(r.page_content)
    print("Metadata:", r.metadata)


###Summary:
**What Is Happening Internally ?**

* Take the user’s original query

* LLM generates multiple rephrased versions of the query

* Perform vector search for each query variation

* Collect results from all searches

* Remove duplicate or repeated documents

* Return the combined set of relevant documents

* User query → multiple rewrites → multiple searches → merged results